# Roman Microlensing Readiness Benchmark -- Colab walkthrough

**Purpose**: demonstrate, end to end on a free Colab CPU runtime, the physical-model validation checks and a small injection-recovery run from the `romanmlr` package.

**Research question**: how do survey cadence, source brightness, and detection thresholds change completeness for Roman-GBTDS-like microlensing searches? (Full statement: `docs/METHODS.md` in the repository.)

**Data terms**: this notebook's default path uses only synthetic injections (no external data download, no credentials). A separate optional cell fetches the public 2018 WFIRST/Roman Microlensing Data Challenge ground-truth tables (small text files, no login required); see `docs/DATA_SOURCES.md`.

**Runtime estimate**: under 2 minutes on the free Colab CPU runtime for the default demonstration grid (40 trials).

**This notebook does not claim any real exoplanet or microlensing detection.** All events analyzed below are synthetic injections with known ground truth.

In [ ]:
%pip install -q git+https://github.com/Biswajit1999/roman-microlensing-readiness.git

import numpy, scipy, pandas, matplotlib
import romanmlr
print("romanmlr", romanmlr.__version__)
print("numpy", numpy.__version__, "scipy", scipy.__version__, "pandas", pandas.__version__, "matplotlib", matplotlib.__version__)

## 1. Physical-model validation, inline

Rather than asking you to trust the README, the two cheapest, highest-value validation checks from the repository's test suite are reproduced here directly.

In [ ]:
import numpy as np
from romanmlr.pspl import magnification_pspl
from romanmlr.fspl import magnification_fspl

# Hand calculation: A(u=1) = (1+2) / (1*sqrt(1+4)) = 3/sqrt(5)
a_hand = 3 / np.sqrt(5)
a_code = magnification_pspl(1.0)
print(f"PSPL A(u=1): hand calculation = {a_hand:.6f}, code = {a_code:.6f}")
assert abs(a_hand - a_code) < 1e-10

# Finite-source magnification must converge to the point-lens value far from the source
rho = 0.01
u = np.array([0.5, 1.0, 2.0])
diff = np.max(np.abs(magnification_fspl(u, rho) - magnification_pspl(u)))
print(f"max |A_FSPL - A_PSPL| for u >> rho: {diff:.2e}")
assert diff < 1e-2
print("Validation checks passed.")

## 2. A small injection-recovery grid

Free-floating-planet (FFP) channel: an isolated point lens, the physically correct treatment for an unbound lens (see `docs/METHODS.md`). We sweep the impact parameter `u0` at fixed, deliberately short/faint parameters chosen to show a real completeness transition (not just a flat 100% or 0% line) within a runtime budget suitable for a free Colab CPU.

In [ ]:
from romanmlr.cadence import CadenceConfig
from romanmlr.injection_recovery import TrialConfig, run_grid
from romanmlr.completeness import completeness_by_bin

SEED_BASE = 0  # fixed for reproducibility

cadence = CadenceConfig(
    n_seasons=2, season_length_days=72, season_gap_days=61,
    cadence_minutes=15, downlink_gap_every_hours=12, random_dropout_frac=0.02,
)

u0_grid = [0.05, 0.3, 0.8, 1.5, 2.5]
n_seeds = 8

configs = [
    TrialConfig(
        channel="ffp", u0=u0, tE=0.15, rho=0.01, t0=30.0,
        mag_ref=23.0, seed=SEED_BASE + seed, cadence=cadence,
    )
    for u0 in u0_grid
    for seed in range(n_seeds)
]

df = run_grid(configs)
df.head()

In [ ]:
comp = completeness_by_bin(df, ["u0"])
comp

## 3. Completeness curve with Wilson-score confidence band

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
yerr = [comp["completeness"] - comp["ci_low"], comp["ci_high"] - comp["completeness"]]
ax.errorbar(comp["u0"], comp["completeness"], yerr=yerr, fmt="o-", capsize=4,
            label=f"n_seeds={n_seeds} per point (Wilson 95% CI)")
ax.set_xlabel("Impact parameter u0 (Einstein radii)")
ax.set_ylabel("Recovery completeness")
ax.set_ylim(-0.05, 1.05)
ax.set_title("FFP injection-recovery completeness vs. impact parameter\n(tE=0.15 d, mag_ref=23.0, demonstration grid only -- not a population result)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Limitations (summary; full list in `docs/LIMITATIONS.md`)

- This is a demonstration grid (`n_seeds=8` per point, one `tE`/brightness pair), not the population-scale result in `configs/default.yaml`.
- The binary-lens (bound-planet) channel is not exercised in this notebook; see `romanmlr.planetary` and its documented ray-shooting resolution floor.
- The noise model is a simple two-term magnitude-dependent model, not an instrument-team-grade exposure-time calculator.
- Detection thresholds (500 for event detection) are literature-typical defaults, not values tuned to this specific noise model.

**Next validation step**: cross-check completeness against an independently implemented microlensing package (`pyLIMA`/`MulensModel`) on a shared test event -- tracked as a repository issue.